# Agentic Evaluation: Evaluating AI Agents Beyond Model Outputs

Previous notebooks evaluated **model outputs** — given a prompt, how good is the response? But modern AI applications increasingly use **agents**: LLMs that reason, select tools, and chain multiple steps to answer complex queries.

Agentic evaluation asks different questions:

- Did the agent choose the **right tools** for the task?
- Were tool call **parameters correct** and well-formed?
- Did the agent **use tool results** faithfully in its final answer?
- Was the agent's **reasoning chain** logical and efficient?

This notebook teaches you how to evaluate an agent's full execution trace — not just its final output.

## What You'll Learn

1. **Build a tool-using healthcare agent** with the [Strands Agents SDK](https://github.com/strands-agents/sdk-python) on Amazon Bedrock
2. **Capture structured execution traces** using Strands hooks (tool calls, results, reasoning)
3. **Define evaluation rubrics** for agent-specific behaviors
4. **Run multi-judge evaluation** on agent traces (LLM-as-a-Jury)
5. **Compute deterministic metrics** (tool selection accuracy, latency, cost)
6. **Visualize agent quality** across multiple dimensions

## From Model Evaluation to Agent Evaluation

| Dimension | Model Evaluation (Notebook 2) | Agent Evaluation (This Notebook) |
|-----------|-------------------------------|----------------------------------|
| **Input** | Single prompt → response | Multi-step execution trace |
| **What's evaluated** | Response quality only | Tool selection + tool use + reasoning + final answer |
| **Metrics** | Correctness, completeness | Tool groundedness, consistency, chaining, answer quality |
| **Failure modes** | Wrong answer | Wrong tool, bad parameters, ignored results, correct tool but wrong answer |

---

## Setup

We import helper functions from `src/utils.py` for Bedrock invocation, RAG retrieval, and evaluation, then initialize the FAISS vector store for our medication knowledge base.

In [ ]:
import json
import time
import numpy as np
import pandas as pd
import concurrent.futures
from typing import Dict, List, Optional
from dataclasses import dataclass, field
import plotly.graph_objects as go
import plotly.express as px

import sys
sys.path.insert(0, '.')

from src.utils import (
    create_bedrock_client,
    initialize_medication_vector_store,
    retrieve_medication_documents_for_query,
    compute_semantic_similarity,
    compute_entity_match_score,
    compute_model_cost,
    evaluate_medical_safety,
    QUALITY_METRICS,
)

# Strands Agents SDK
from strands import Agent, tool
from strands.models import BedrockModel
from strands.hooks.registry import HookProvider, HookRegistry
from strands.hooks.events import (
    BeforeInvocationEvent,
    AfterInvocationEvent,
    BeforeToolCallEvent,
    AfterToolCallEvent,
    AfterModelCallEvent,
)

REGION = "us-east-1"
bedrock = create_bedrock_client(REGION)

# Agent model — the model powering our healthcare agent
AGENT_MODEL_ID = "us.anthropic.claude-3-5-haiku-20241022-v1:0"

# Judge models for rubric evaluation (LLM-as-a-Jury)
JUDGE_MODEL_IDS = [
    "us.anthropic.claude-3-5-haiku-20241022-v1:0",
    "us.amazon.nova-pro-v1:0",
]

print(f"Region:       {REGION}")
print(f"Agent model:  {AGENT_MODEL_ID}")
print(f"Judge models: {len(JUDGE_MODEL_IDS)}")

In [ ]:
# Initialize FAISS vector store for medication RAG retrieval
print("Initializing FAISS vector store...")
vector_store = initialize_medication_vector_store()
print("Vector store ready.")

---

## Part 1: Build a Tool-Using Healthcare Agent

We build a healthcare agent using the **Strands Agents SDK** on Amazon Bedrock. The agent has two tools:

| Tool | Purpose | Backed by |
|------|---------|----------|
| `medication_lookup` | Look up medication info (side effects, dosing, warnings) | FAISS vector store + sentence-transformers |
| `drug_interaction_check` | Check interactions between medications | Curated interaction database |

### Strands Agent Architecture

With Strands, the agent loop is handled automatically by the SDK:

1. **`@tool` decorated functions** define the tools — docstrings and type hints are converted to JSON schema automatically
2. **`Agent`** manages the conversation loop — sends the user query, routes tool calls, and collects the final answer
3. **`HookProvider`** captures a structured execution trace via lifecycle callbacks (`BeforeToolCall`, `AfterToolCall`, `AfterModelCall`, etc.)

This replaces the manual `bedrock.converse()` loop with a declarative agent definition while producing the exact same trace format for evaluation.

In [ ]:
# ── Agent tools (Strands @tool decorator) ──

AGENT_SYSTEM_PROMPT = """You are a healthcare assistant with access to a medication database and a drug interaction checker.
Use the medication_lookup tool when the user asks about a specific drug's side effects, dosing, or warnings.
Use the drug_interaction_check tool when the user asks about combining medications or potential interactions.
You may call multiple tools if the question requires both lookup and interaction checking.
Always base your final answer on the tool results. Be accurate and concise."""

# ── Drug interaction knowledge base ──

INTERACTION_DB = {
    ("aspirin", "warfarin"): {
        "severity": "high",
        "description": "Increased risk of bleeding when combining anticoagulant with antiplatelet agent.",
        "recommendation": "Avoid combination unless specifically directed by physician. Monitor for signs of bleeding."
    },
    ("lisinopril", "potassium"): {
        "severity": "high",
        "description": "ACE inhibitors increase potassium retention. Adding potassium supplements risks hyperkalemia.",
        "recommendation": "Avoid potassium supplements unless directed by physician. Monitor potassium levels."
    },
    ("lisinopril", "metformin"): {
        "severity": "low",
        "description": "Both may affect blood sugar levels. Minor interaction.",
        "recommendation": "Monitor blood glucose; generally safe to combine."
    },
    ("metformin", "warfarin"): {
        "severity": "moderate",
        "description": "Metformin may enhance the anticoagulant effect of warfarin.",
        "recommendation": "Monitor INR levels closely when starting or adjusting metformin."
    },
    ("aspirin", "metformin"): {
        "severity": "low",
        "description": "No significant interaction. Both commonly prescribed together in diabetic patients.",
        "recommendation": "Safe to combine. No special monitoring required."
    },
}


@tool
def medication_lookup(medication_name: str) -> dict:
    """Look up information about a medication including side effects, dosing, and warnings. Use this when the user asks about a specific drug."""
    query = f"What are the side effects and important information about {medication_name}?"
    context, rag_result = retrieve_medication_documents_for_query(query)
    return {
        "medication": medication_name,
        "information": context[:1500],
        "source_document": rag_result.document_id,
        "relevance_score": round(rag_result.relevance_score, 3)
    }


@tool
def drug_interaction_check(medications: list) -> dict:
    """Check for interactions between two or more medications. Use this when the user asks about combining drugs or potential interactions."""
    interactions = []
    meds_lower = [m.lower().strip() for m in medications]

    for i in range(len(meds_lower)):
        for j in range(i + 1, len(meds_lower)):
            pair = tuple(sorted([meds_lower[i], meds_lower[j]]))
            if pair in INTERACTION_DB:
                entry = INTERACTION_DB[pair]
                interactions.append({
                    "drug_pair": f"{meds_lower[i]} + {meds_lower[j]}",
                    "severity": entry["severity"],
                    "description": entry["description"],
                    "recommendation": entry["recommendation"]
                })

    if not interactions:
        interactions.append({
            "drug_pair": " + ".join(meds_lower),
            "severity": "none",
            "description": "No known interactions found in database.",
            "recommendation": "Always consult your physician or pharmacist for a complete interaction check."
        })

    return {"medications_checked": medications, "interactions": interactions}


print("Agent tools defined:")
print(f"  - medication_lookup: {medication_lookup.tool_spec['description'][:60]}...")
print(f"  - drug_interaction_check: {drug_interaction_check.tool_spec['description'][:60]}...")

In [ ]:
# ── Strands Agent with trace capture via hooks ──

class AgentTraceHook:
    """HookProvider that captures structured execution traces for evaluation."""

    def __init__(self, query: str, model_id: str):
        self.trace = {
            "user_query": query,
            "model_id": model_id,
            "steps": [],
            "tools_used": [],
            "final_answer": None,
            "total_latency_ms": 0,
            "total_input_tokens": 0,
            "total_output_tokens": 0,
            "num_turns": 0,
        }
        self._start_time = None
        self._tool_starts = {}
        self._step_counter = 0

    def register_hooks(self, registry: HookRegistry):
        registry.add_callback(BeforeInvocationEvent, self._on_before_invocation)
        registry.add_callback(BeforeToolCallEvent, self._on_before_tool_call)
        registry.add_callback(AfterToolCallEvent, self._on_after_tool_call)
        registry.add_callback(AfterModelCallEvent, self._on_after_model_call)
        registry.add_callback(AfterInvocationEvent, self._on_after_invocation)

    def _on_before_invocation(self, event: BeforeInvocationEvent):
        self._start_time = time.time()

    def _on_before_tool_call(self, event: BeforeToolCallEvent):
        tool_use_id = event.tool_use["toolUseId"]
        self._tool_starts[tool_use_id] = time.time()

    def _on_after_tool_call(self, event: AfterToolCallEvent):
        tool_use_id = event.tool_use["toolUseId"]
        tool_name = event.tool_use["name"]
        tool_input = event.tool_use["input"]

        start = self._tool_starts.pop(tool_use_id, time.time())
        latency_ms = round((time.time() - start) * 1000, 1)

        tool_result = {}
        for content_item in event.result.get("content", []):
            if "json" in content_item:
                tool_result = content_item["json"]
                break
            elif "text" in content_item:
                tool_result = {"text": content_item["text"]}
                break

        self._step_counter += 1
        self.trace["steps"].append({
            "step": self._step_counter,
            "kind": "TOOL_CALL",
            "tool_name": tool_name,
            "tool_input": tool_input,
            "tool_result": tool_result,
            "latency_ms": latency_ms,
        })

        if tool_name not in self.trace["tools_used"]:
            self.trace["tools_used"].append(tool_name)

    def _on_after_model_call(self, event: AfterModelCallEvent):
        self.trace["num_turns"] += 1

    def _on_after_invocation(self, event: AfterInvocationEvent):
        total_ms = round((time.time() - self._start_time) * 1000, 1) if self._start_time else 0
        self.trace["total_latency_ms"] = total_ms

        result = event.result
        # Extract final answer from result message
        if result and result.message:
            text_parts = []
            for block in result.message.get("content", []):
                if "text" in block:
                    text_parts.append(block["text"])
            if text_parts:
                self.trace["final_answer"] = "\n".join(text_parts)
                self._step_counter += 1
                self.trace["steps"].append({
                    "step": self._step_counter,
                    "kind": "FINAL_ANSWER",
                    "content": self.trace["final_answer"],
                    "latency_ms": total_ms,
                })

        # Extract token usage from metrics
        if result and result.metrics:
            usage = result.metrics.accumulated_usage
            self.trace["total_input_tokens"] = usage.get("inputTokens", 0)
            self.trace["total_output_tokens"] = usage.get("outputTokens", 0)


def run_agent(query: str, model_id: str = AGENT_MODEL_ID) -> dict:
    """Run the healthcare agent on a query, capturing a full execution trace via hooks."""
    hook = AgentTraceHook(query, model_id)
    model = BedrockModel(model_id=model_id, region_name=REGION, max_tokens=1024)
    agent = Agent(
        model=model,
        tools=[medication_lookup, drug_interaction_check],
        system_prompt=AGENT_SYSTEM_PROMPT,
        hooks=[hook],
        callback_handler=None,
    )
    result = agent(query)
    return hook.trace


print("Strands Agent with trace hooks defined. Ready to run.")

---

### Test Queries

We define 5 healthcare queries that exercise different tool-use patterns. Each has:
- **Expected tools** — which tools the agent *should* call
- **Golden answer** — reference answer for quality evaluation
- **Critical entities** — medical terms that must appear in a safe response

In [ ]:
AGENT_TEST_CASES = [
    {
        "id": "AGT-001",
        "query": "What are the common side effects of metformin?",
        "expected_tools": ["medication_lookup"],
        "golden_answer": "Common side effects of metformin include nausea, diarrhea, stomach upset, metallic taste, and vitamin B12 deficiency. Rare but serious: lactic acidosis.",
        "critical_entities": ["nausea", "diarrhea", "lactic acidosis"]
    },
    {
        "id": "AGT-002",
        "query": "Can I safely take warfarin and aspirin together?",
        "expected_tools": ["drug_interaction_check"],
        "golden_answer": "Combining warfarin and aspirin significantly increases bleeding risk. This combination should be avoided unless specifically directed by your physician.",
        "critical_entities": ["bleeding", "risk"]
    },
    {
        "id": "AGT-003",
        "query": "I take metformin for diabetes. My doctor wants to add lisinopril for blood pressure. What should I know about lisinopril and are there any interactions between these medications?",
        "expected_tools": ["medication_lookup", "drug_interaction_check"],
        "golden_answer": "Lisinopril side effects include dry cough, dizziness, and headache. Regarding interactions with metformin: both may affect blood sugar levels (minor interaction). Monitor glucose levels.",
        "critical_entities": ["cough", "dizziness", "blood sugar"]
    },
    {
        "id": "AGT-004",
        "query": "What are the warning signs of low blood sugar I should watch for while on diabetes medication?",
        "expected_tools": ["medication_lookup"],
        "golden_answer": "Warning signs of low blood sugar include shakiness, sweating, rapid heartbeat, confusion, dizziness, and hunger. Severe cases may cause seizures or loss of consciousness.",
        "critical_entities": ["sweating", "confusion", "dizziness"]
    },
    {
        "id": "AGT-005",
        "query": "I'm on metformin and lisinopril. My doctor suggested adding a potassium supplement. Should I be concerned about any interactions?",
        "expected_tools": ["drug_interaction_check"],
        "golden_answer": "Yes — lisinopril (ACE inhibitor) increases potassium retention. Adding potassium supplements risks hyperkalemia. Avoid unless directed and monitored by your physician.",
        "critical_entities": ["potassium", "hyperkalemia"]
    }
]

print(f"Test cases: {len(AGENT_TEST_CASES)}")
print(f"\nTool-use patterns:")
for tc in AGENT_TEST_CASES:
    print(f"  {tc['id']}: {' + '.join(tc['expected_tools']):<45} {tc['query'][:55]}...")

---

### Run the Agent

We run the agent on each test query and capture the full execution trace. Each trace records the user query, every tool call with its parameters and results, and the final answer.

In [ ]:
# Run the agent on all test cases
traces = {}

print("Running agent on test queries...")
print("=" * 80)

for tc in AGENT_TEST_CASES:
    print(f"\n{tc['id']}: {tc['query'][:60]}...")
    trace = run_agent(tc["query"])
    traces[tc["id"]] = trace

    tool_names = [s["tool_name"] for s in trace["steps"] if s["kind"] == "TOOL_CALL"]
    print(f"  Tools called: {tool_names}")
    print(f"  Turns: {trace['num_turns']} | Latency: {trace['total_latency_ms']:.0f}ms | Tokens: {trace['total_input_tokens']}in/{trace['total_output_tokens']}out")
    print(f"  Answer: {trace['final_answer'][:100]}..." if trace['final_answer'] and len(trace['final_answer']) > 100 else f"  Answer: {trace['final_answer']}")

print("\n" + "=" * 80)
print(f"All {len(traces)} traces captured.")

---

## Part 2: Inspect an Agent Trace

Before evaluation, let's inspect what a trace looks like. A trace is the **complete record** of the agent's execution — the unit of evaluation in agentic systems.

This is analogous to the normalized trace format used by production agent evaluation frameworks like the `agent-eval` tool in this repository, which normalizes arbitrary trace formats into a standard schema for offline evaluation.

In [ ]:
# Display a single trace (AGT-003 — multi-tool query)
sample_trace = traces["AGT-003"]

print("AGENT TRACE: AGT-003")
print("=" * 80)
print(f"User Query: {sample_trace['user_query']}")
print(f"Model: {sample_trace['model_id']}")
print(f"Total Turns: {sample_trace['num_turns']}")
print(f"Total Latency: {sample_trace['total_latency_ms']:.0f}ms")
print(f"Tools Used: {sample_trace['tools_used']}")
print()

for step in sample_trace["steps"]:
    if step["kind"] == "TOOL_CALL":
        print(f"  Step {step['step']} [TOOL_CALL] {step['tool_name']}")
        print(f"    Input:  {json.dumps(step['tool_input'])}")
        result_preview = json.dumps(step['tool_result'])[:200]
        print(f"    Result: {result_preview}...")
        print(f"    Latency: {step['latency_ms']}ms")
    elif step["kind"] == "FINAL_ANSWER":
        print(f"  Step {step['step']} [FINAL_ANSWER]")
        print(f"    {step['content'][:300]}")
    print()

---

## Part 3: Define Evaluation Rubrics

In model evaluation (Notebook 2), we scored responses on generic quality metrics (correctness, completeness, etc.). For **agent evaluation**, we need rubrics that assess agent-specific behaviors.

These rubrics are inspired by the `agent-eval` framework's [default rubrics](../../agent-eval/agent_eval/evaluators/trace_eval/default_rubrics.yaml), which define production-grade evaluation criteria for agent traces.

| Rubric | What It Measures | Evidence Used |
|--------|-----------------|---------------|
| **Tool Groundedness** | Are tool calls aligned with the user query? | User query + tool calls |
| **Tool Consistency** | Are tool results faithfully reflected in the answer? | Tool results + final answer |
| **Tool Call Quality** | Are tool parameters correct and well-formed? | Tool call parameters |
| **Answer Quality** | Is the final answer correct, complete, and safe? | Full trace + golden answer |

In [ ]:
# ── Evaluation rubrics ──

AGENT_RUBRICS = [
    {
        "rubric_id": "TOOL_GROUNDEDNESS",
        "description": "Are tool calls grounded in the user query?",
        "weight": 1.0,
        "instructions": """Score 1-5 based on how well tool calls align with the user's query:
5 = All tool calls directly address user needs with appropriate parameters
4 = Most tool calls are relevant with minor parameter issues
3 = Some tool calls are relevant but others are tangential
2 = Few tool calls address the user query directly
1 = Tool calls are unrelated or contradict user intent"""
    },
    {
        "rubric_id": "TOOL_CONSISTENCY",
        "description": "Are tool results used consistently in the final answer?",
        "weight": 1.0,
        "instructions": """Score 1-5 based on how consistently tool results are incorporated:
5 = All tool results are accurately reflected in the final answer
4 = Most tool results are used correctly with minor omissions
3 = Some tool results are used but others are ignored or misrepresented
2 = Few tool results are incorporated into the final answer
1 = Tool results are contradicted or completely ignored"""
    },
    {
        "rubric_id": "TOOL_CALL_QUALITY",
        "description": "Are tool call parameters correct and well-formed?",
        "weight": 1.0,
        "instructions": """Score 1-5 based on tool call parameter quality:
5 = All parameters are correct, complete, and optimally formatted
4 = Parameters are mostly correct with minor formatting issues
3 = Some parameters are correct but others are incomplete or suboptimal
2 = Many parameters are incorrect or missing required fields
1 = Parameters are fundamentally wrong or would cause tool failures"""
    },
    {
        "rubric_id": "ANSWER_QUALITY",
        "description": "Is the final answer correct, complete, and medically safe?",
        "weight": 1.5,
        "instructions": """Score 1-5 based on overall answer quality:
5 = Accurate, complete, well-structured, cites tool evidence, includes safety caveats
4 = Mostly accurate with minor gaps but no safety issues
3 = Acceptable accuracy but missing important details
2 = Significant errors or omissions, potential safety concerns
1 = Incorrect, dangerous, or completely unhelpful"""
    }
]

print("Evaluation rubrics:")
for r in AGENT_RUBRICS:
    print(f"  {r['rubric_id']:<22} (weight: {r['weight']}) — {r['description']}")

In [ ]:
# ── Evidence extraction ──
# Each rubric evaluates different parts of the trace.
# We extract only the relevant evidence, similar to the JSONPath selectors
# used by the agent-eval framework's evidence extractor.

def extract_evidence(trace: dict, rubric_id: str, golden_answer: str = None) -> str:
    """Extract rubric-specific evidence from an agent trace."""

    if rubric_id == "TOOL_GROUNDEDNESS":
        evidence = f"User Query: {trace['user_query']}\n\nTool Calls Made:\n"
        for step in trace["steps"]:
            if step["kind"] == "TOOL_CALL":
                evidence += f"- {step['tool_name']}({json.dumps(step['tool_input'])})\n"
        if not any(s["kind"] == "TOOL_CALL" for s in trace["steps"]):
            evidence += "- (No tool calls made)\n"
        return evidence

    elif rubric_id == "TOOL_CONSISTENCY":
        evidence = "Tool Results Returned:\n"
        for step in trace["steps"]:
            if step["kind"] == "TOOL_CALL":
                result_str = json.dumps(step["tool_result"], indent=2)[:800]
                evidence += f"- {step['tool_name']}: {result_str}\n\n"
        evidence += f"Final Answer:\n{trace.get('final_answer', '(No final answer)')}"
        return evidence

    elif rubric_id == "TOOL_CALL_QUALITY":
        evidence = "Tool Calls with Parameters:\n"
        for step in trace["steps"]:
            if step["kind"] == "TOOL_CALL":
                evidence += f"- Tool: {step['tool_name']}\n"
                evidence += f"  Parameters: {json.dumps(step['tool_input'], indent=2)}\n"
        return evidence

    elif rubric_id == "ANSWER_QUALITY":
        evidence = f"User Query: {trace['user_query']}\n\n"
        evidence += "Agent Execution:\n"
        for step in trace["steps"]:
            if step["kind"] == "TOOL_CALL":
                result_str = json.dumps(step["tool_result"])[:400]
                evidence += f"  -> Called {step['tool_name']}({json.dumps(step['tool_input'])})\n"
                evidence += f"  <- Result: {result_str}\n"
        evidence += f"\nFinal Answer:\n{trace.get('final_answer', '(No answer)')}"
        if golden_answer:
            evidence += f"\n\nReference Answer:\n{golden_answer}"
        return evidence

    return ""


# Preview evidence for one trace and one rubric
print("EVIDENCE PREVIEW: TOOL_GROUNDEDNESS for AGT-003")
print("=" * 80)
print(extract_evidence(traces["AGT-003"], "TOOL_GROUNDEDNESS"))

---

## Part 4: LLM-as-a-Jury on Agent Traces

We apply the same **multi-judge evaluation** approach from Notebook 2, but now evaluate **execution traces against rubrics** rather than plain responses against quality metrics.

For each (trace, rubric) pair:
1. Extract rubric-specific evidence from the trace
2. Build a judge prompt with the evidence and scoring instructions
3. Send to multiple judge models in parallel
4. Aggregate scores using majority voting

In [ ]:
import re

def build_rubric_judge_prompt(evidence: str, rubric: dict) -> str:
    """Build a judge prompt for evaluating a trace against a specific rubric."""
    return f"""You are evaluating an AI healthcare agent's execution trace against a specific rubric.

RUBRIC: {rubric['rubric_id']} — {rubric['description']}

SCORING INSTRUCTIONS:
{rubric['instructions']}

EVIDENCE FROM AGENT TRACE:
{evidence}

Evaluate the evidence against the rubric. Respond with ONLY valid JSON:
{{{{
    "score": <integer 1-5>,
    "judgment": "PASS" or "FAIL",
    "explanation": "<one sentence explanation>"
}}}}

A score of 3 or higher is PASS."""


def evaluate_trace_with_judge(evidence: str, rubric: dict, judge_model_id: str) -> dict:
    """Evaluate a trace against a rubric using a single judge."""
    prompt = build_rubric_judge_prompt(evidence, rubric)
    try:
        response = bedrock.converse(
            modelId=judge_model_id,
            messages=[{"role": "user", "content": [{"text": prompt}]}],
            system=[{"text": "You are an expert evaluator. Respond only with valid JSON."}],
            inferenceConfig={"maxTokens": 300, "temperature": 0.1}
        )
        response_text = response["output"]["message"]["content"][0]["text"]
        json_match = re.search(r'\{[\s\S]*\}', response_text)
        if json_match:
            result = json.loads(json_match.group())
            return {
                "score": result.get("score", 1),
                "judgment": result.get("judgment", "FAIL"),
                "explanation": result.get("explanation", ""),
                "judge_model": judge_model_id
            }
    except Exception as e:
        pass
    return {"score": 1, "judgment": "FAIL", "explanation": f"Judge error", "judge_model": judge_model_id}


def evaluate_trace_with_jury(
    trace: dict, rubric: dict, judge_model_ids: list, golden_answer: str = None
) -> dict:
    """Evaluate a trace against a rubric using multiple judges (jury)."""
    evidence = extract_evidence(trace, rubric["rubric_id"], golden_answer)

    # Run judges in parallel
    judge_results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=len(judge_model_ids)) as executor:
        futures = {
            executor.submit(evaluate_trace_with_judge, evidence, rubric, jid): jid
            for jid in judge_model_ids
        }
        for future in concurrent.futures.as_completed(futures):
            judge_results.append(future.result())

    # Aggregate: majority vote for judgment, median for score
    scores = [r["score"] for r in judge_results]
    pass_votes = sum(1 for r in judge_results if r["judgment"].upper() == "PASS")

    return {
        "rubric_id": rubric["rubric_id"],
        "median_score": float(np.median(scores)),
        "mean_score": float(np.mean(scores)),
        "majority_judgment": "PASS" if pass_votes > len(judge_results) / 2 else "FAIL",
        "judge_results": judge_results
    }


print("Judge evaluation functions defined.")

In [ ]:
# Run rubric evaluation for all traces
evaluation_results = {}  # test_id -> {rubric_id -> result}

print("Running LLM-as-a-Jury rubric evaluation...")
print("=" * 80)

for tc in AGENT_TEST_CASES:
    test_id = tc["id"]
    trace = traces[test_id]
    evaluation_results[test_id] = {}

    print(f"\n{test_id}: {tc['query'][:50]}...")

    for rubric in AGENT_RUBRICS:
        result = evaluate_trace_with_jury(
            trace=trace,
            rubric=rubric,
            judge_model_ids=JUDGE_MODEL_IDS,
            golden_answer=tc.get("golden_answer")
        )
        evaluation_results[test_id][rubric["rubric_id"]] = result

        votes = [r["judgment"][0] for r in result["judge_results"]]
        status = "PASS" if result["majority_judgment"] == "PASS" else "FAIL"
        icon = "\u2713" if status == "PASS" else "\u2717"
        print(f"  {icon} {rubric['rubric_id']:<22} {result['median_score']:.1f}/5  {status}  (votes: {'/'.join(votes)})")

print("\n" + "=" * 80)
print("Rubric evaluation complete.")

---

## Part 5: Deterministic Metrics

Not all evaluation requires an LLM judge. **Deterministic metrics** are computed directly from the trace without any LLM calls — they're fast, free, and reproducible.

| Metric | How It's Computed | Why It Matters |
|--------|------------------|----------------|
| **Tool selection accuracy** | Compare actual vs expected tools | Measures routing correctness |
| **Semantic similarity** | Cosine similarity to golden answer | Measures answer quality (fast proxy) |
| **Entity match score** | Critical medical entities present | Measures safety |
| **Latency** | Total execution time | Measures user experience |
| **Cost** | Token counts × pricing | Measures efficiency |

In [ ]:
# Compute deterministic metrics for all traces
deterministic_metrics = {}

print("Computing deterministic metrics...")
print("=" * 80)

for tc in AGENT_TEST_CASES:
    test_id = tc["id"]
    trace = traces[test_id]
    final_answer = trace.get("final_answer", "")

    # Tool selection accuracy
    actual_tools = sorted(trace["tools_used"])
    expected_tools = sorted(tc["expected_tools"])
    tool_match = actual_tools == expected_tools

    # Semantic similarity to golden answer
    similarity = compute_semantic_similarity(final_answer, tc["golden_answer"]) if final_answer else 0.0

    # Entity match (medical safety)
    entity_score, found, missing = compute_entity_match_score(
        final_answer, tc["critical_entities"]
    ) if final_answer else (0.0, [], tc["critical_entities"])

    # Safety evaluation
    safety = evaluate_medical_safety(
        final_answer, tc["critical_entities"]
    ) if final_answer else None

    # Cost estimate
    cost = compute_model_cost(
        trace["total_input_tokens"], trace["total_output_tokens"], AGENT_MODEL_ID
    )

    deterministic_metrics[test_id] = {
        "tool_match": tool_match,
        "actual_tools": actual_tools,
        "expected_tools": expected_tools,
        "semantic_similarity": similarity,
        "entity_match_score": entity_score,
        "entities_found": found,
        "entities_missing": missing,
        "safety_pass": safety.safety_pass if safety else False,
        "latency_ms": trace["total_latency_ms"],
        "total_tokens": trace["total_input_tokens"] + trace["total_output_tokens"],
        "cost": cost,
        "num_tool_calls": sum(1 for s in trace["steps"] if s["kind"] == "TOOL_CALL"),
        "num_turns": trace["num_turns"],
    }

    icon = "\u2713" if tool_match else "\u2717"
    print(f"\n{test_id}: {icon} Tool match | Similarity: {similarity:.2f} | Entities: {entity_score:.0%} | Latency: {trace['total_latency_ms']:.0f}ms")
    if not tool_match:
        print(f"  Expected: {expected_tools} | Actual: {actual_tools}")
    if missing:
        print(f"  Missing entities: {missing}")

# Summary
tool_accuracy = sum(1 for m in deterministic_metrics.values() if m["tool_match"]) / len(deterministic_metrics)
avg_similarity = np.mean([m["semantic_similarity"] for m in deterministic_metrics.values()])
avg_entity = np.mean([m["entity_match_score"] for m in deterministic_metrics.values()])
avg_latency = np.mean([m["latency_ms"] for m in deterministic_metrics.values()])
total_cost = sum(m["cost"] for m in deterministic_metrics.values())

print("\n" + "=" * 80)
print("DETERMINISTIC METRICS SUMMARY")
print(f"  Tool Selection Accuracy: {tool_accuracy:.0%}")
print(f"  Avg Semantic Similarity: {avg_similarity:.2f}")
print(f"  Avg Entity Match Score:  {avg_entity:.0%}")
print(f"  Avg Latency:             {avg_latency:.0f}ms")
print(f"  Total Cost:              ${total_cost:.6f}")

---

## Part 6: Visualize Agent Quality

We combine LLM-judged rubric scores and deterministic metrics into interactive visualizations — the same dark Plotly theme used in Notebook 2.

### Rubric Score Heatmap

In [ ]:
DARK_BG = "#161e2d"
GRID_COLOR = "#2a3a5a"
TEXT_COLOR = "#e0e0e0"

# Build heatmap data: rows = test cases, columns = rubrics
test_ids = [tc["id"] for tc in AGENT_TEST_CASES]
rubric_ids = [r["rubric_id"] for r in AGENT_RUBRICS]

z_data = []
for test_id in test_ids:
    row = [evaluation_results[test_id][rid]["median_score"] for rid in rubric_ids]
    z_data.append(row)

custom_colorscale = [
    [0.0, '#8B0000'],
    [0.25, '#DC143C'],
    [0.5, '#FF8C00'],
    [0.75, '#32CD32'],
    [1.0, '#006400']
]

fig = go.Figure(data=go.Heatmap(
    z=z_data,
    x=[rid.replace('_', ' ').title() for rid in rubric_ids],
    y=test_ids,
    colorscale=custom_colorscale,
    zmin=1, zmax=5,
    text=[[f"{val:.1f}" for val in row] for row in z_data],
    texttemplate="%{text}",
    textfont={"size": 13, "color": "white"},
    hovertemplate="Test: %{y}<br>Rubric: %{x}<br>Score: %{z:.1f}<extra></extra>",
    colorbar=dict(title="Score", tickvals=[1, 2, 3, 4, 5])
))

fig.update_layout(
    title=dict(text="Agent Rubric Scores by Test Case", font=dict(size=18, color=TEXT_COLOR)),
    template="plotly_dark",
    paper_bgcolor=DARK_BG,
    plot_bgcolor=DARK_BG,
    height=350,
    margin=dict(l=80, r=50, t=80, b=80)
)

fig.show()

### Radar Chart: Multi-Dimensional Agent Quality

In [ ]:
# Radar chart combining rubric scores (normalized) and deterministic metrics
radar_dims = ['Tool Groundedness', 'Tool Consistency', 'Tool Call Quality',
              'Answer Quality', 'Tool Accuracy', 'Semantic Similarity', 'Safety']

# Compute per-test-case values
colors = px.colors.qualitative.Set2

fig = go.Figure()

for i, test_id in enumerate(test_ids):
    ev = evaluation_results[test_id]
    dm = deterministic_metrics[test_id]

    values = [
        ev["TOOL_GROUNDEDNESS"]["median_score"] / 5,
        ev["TOOL_CONSISTENCY"]["median_score"] / 5,
        ev["TOOL_CALL_QUALITY"]["median_score"] / 5,
        ev["ANSWER_QUALITY"]["median_score"] / 5,
        1.0 if dm["tool_match"] else 0.0,
        dm["semantic_similarity"],
        dm["entity_match_score"],
    ]
    values.append(values[0])

    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=radar_dims + [radar_dims[0]],
        fill='toself',
        name=test_id,
        line=dict(color=colors[i % len(colors)]),
        opacity=0.65
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 1], tickvals=[0.2, 0.4, 0.6, 0.8, 1.0], gridcolor=GRID_COLOR),
        angularaxis=dict(gridcolor=GRID_COLOR)
    ),
    title=dict(text="Agent Quality Radar — Per Test Case", font=dict(size=18, color=TEXT_COLOR)),
    template="plotly_dark",
    paper_bgcolor=DARK_BG,
    showlegend=True,
    legend=dict(x=1.1, y=0.5),
    height=500
)

fig.show()

### Tool Selection Accuracy

In [ ]:
# Tool selection analysis — confusion-matrix style
all_tool_names = sorted({t for tc in AGENT_TEST_CASES for t in tc["expected_tools"]} |
                        {t for m in deterministic_metrics.values() for t in m["actual_tools"]})

print("TOOL SELECTION ANALYSIS")
print("=" * 80)
print(f"{'Test ID':<10} {'Expected':<40} {'Actual':<40} {'Match'}")
print("-" * 80)

for tc in AGENT_TEST_CASES:
    dm = deterministic_metrics[tc["id"]]
    icon = "\u2713" if dm["tool_match"] else "\u2717"
    print(f"{tc['id']:<10} {str(dm['expected_tools']):<40} {str(dm['actual_tools']):<40} {icon}")

print(f"\nOverall Tool Selection Accuracy: {tool_accuracy:.0%}")

# Bar chart: expected vs actual tool usage counts
expected_counts = {}
actual_counts = {}
for tc in AGENT_TEST_CASES:
    dm = deterministic_metrics[tc["id"]]
    for t in dm["expected_tools"]:
        expected_counts[t] = expected_counts.get(t, 0) + 1
    for t in dm["actual_tools"]:
        actual_counts[t] = actual_counts.get(t, 0) + 1

fig = go.Figure()
fig.add_trace(go.Bar(name='Expected', x=all_tool_names, y=[expected_counts.get(t, 0) for t in all_tool_names],
                     marker_color='#4a9eff'))
fig.add_trace(go.Bar(name='Actual', x=all_tool_names, y=[actual_counts.get(t, 0) for t in all_tool_names],
                     marker_color='#32CD32'))

fig.update_layout(
    title=dict(text="Tool Usage: Expected vs Actual", font=dict(size=18, color=TEXT_COLOR)),
    barmode='group',
    template="plotly_dark",
    paper_bgcolor=DARK_BG,
    plot_bgcolor=DARK_BG,
    yaxis=dict(title="Count", gridcolor=GRID_COLOR, dtick=1),
    xaxis=dict(title="Tool"),
    height=350
)

fig.show()

### Agent Evaluation Summary Table

In [ ]:
# Summary table combining rubric scores and deterministic metrics
header = ['Test ID', 'Tool Ground.', 'Tool Consist.', 'Call Quality', 'Answer Quality',
          'Tool Match', 'Similarity', 'Safety', 'Latency (ms)', 'Cost ($)']

cell_values = [[] for _ in header]
fill_colors = []

for test_id in test_ids:
    ev = evaluation_results[test_id]
    dm = deterministic_metrics[test_id]

    cell_values[0].append(test_id)
    cell_values[1].append(f"{ev['TOOL_GROUNDEDNESS']['median_score']:.1f}/5")
    cell_values[2].append(f"{ev['TOOL_CONSISTENCY']['median_score']:.1f}/5")
    cell_values[3].append(f"{ev['TOOL_CALL_QUALITY']['median_score']:.1f}/5")
    cell_values[4].append(f"{ev['ANSWER_QUALITY']['median_score']:.1f}/5")
    cell_values[5].append("\u2713" if dm["tool_match"] else "\u2717")
    cell_values[6].append(f"{dm['semantic_similarity']:.2f}")
    cell_values[7].append(f"{dm['entity_match_score']:.0%}")
    cell_values[8].append(f"{dm['latency_ms']:.0f}")
    cell_values[9].append(f"${dm['cost']:.6f}")

    # Color rows by overall pass/fail
    all_pass = all(ev[rid]["majority_judgment"] == "PASS" for rid in rubric_ids)
    fill_colors.append('#1a3a1a' if all_pass else '#3a1a1a')

fig = go.Figure(data=[go.Table(
    header=dict(
        values=[f"<b>{h}</b>" for h in header],
        fill_color='#0d1520',
        align='center',
        font=dict(color=TEXT_COLOR, size=12),
        height=32
    ),
    cells=dict(
        values=cell_values,
        fill_color=[fill_colors for _ in header],
        align='center',
        font=dict(color=TEXT_COLOR, size=11),
        height=28
    )
)])

fig.update_layout(
    title=dict(text="Agent Evaluation Summary", font=dict(size=18, color=TEXT_COLOR)),
    paper_bgcolor=DARK_BG,
    height=280,
    margin=dict(l=20, r=20, t=60, b=20)
)

fig.show()

In [ ]:
# Final scorecard
print("=" * 80)
print("                    AGENT EVALUATION SCORECARD")
print("=" * 80)
print(f"\nAgent Model: {AGENT_MODEL_ID}")
print(f"Test Cases:  {len(AGENT_TEST_CASES)}")
print(f"Judges:      {len(JUDGE_MODEL_IDS)}")
print()

# Aggregate rubric scores
print("RUBRIC SCORES (averaged across all test cases):")
print("-" * 60)
weighted_total = 0
total_weight = 0

for rubric in AGENT_RUBRICS:
    rid = rubric["rubric_id"]
    scores = [evaluation_results[tid][rid]["median_score"] for tid in test_ids]
    avg = np.mean(scores)
    pass_rate = sum(1 for tid in test_ids if evaluation_results[tid][rid]["majority_judgment"] == "PASS") / len(test_ids)
    bar = "\u2588" * int(avg) + "\u2591" * (5 - int(avg))
    print(f"  {rid:<22} {bar} {avg:.2f}/5  (Pass: {pass_rate:.0%})")
    weighted_total += avg * rubric["weight"]
    total_weight += rubric["weight"]

weighted_avg = weighted_total / total_weight
print(f"\n  Weighted Average:            {weighted_avg:.2f}/5")

print("\nDETERMINISTIC METRICS:")
print("-" * 60)
print(f"  Tool Selection Accuracy:  {tool_accuracy:.0%}")
print(f"  Avg Semantic Similarity:  {avg_similarity:.2f}")
print(f"  Avg Entity Match (Safety):{avg_entity:.0%}")
print(f"  Avg Latency:              {avg_latency:.0f}ms")
print(f"  Total Cost ({len(AGENT_TEST_CASES)} queries):    ${total_cost:.6f}")

# Overall verdict
overall_pass = weighted_avg >= 3.0 and tool_accuracy >= 0.6 and avg_entity >= 0.6
print("\n" + "=" * 80)
verdict = "PASS" if overall_pass else "FAIL"
print(f"OVERALL VERDICT: {verdict}")
print("=" * 80)

---

## Summary: Agentic Evaluation Checklist

### Before Evaluation

- [ ] Define agent tools and their expected behaviors
- [ ] Create test queries with expected tool selections and golden answers
- [ ] Identify critical entities for safety checks
- [ ] Select evaluation rubrics (tool groundedness, consistency, quality, answer quality)
- [ ] Configure judge models (2+ for robust consensus)

### During Evaluation

- [ ] Run agent on all test queries
- [ ] Capture structured execution traces (every tool call, result, and final answer)
- [ ] Extract rubric-specific evidence from traces
- [ ] Run LLM-as-a-Jury evaluation per rubric
- [ ] Compute deterministic metrics (tool accuracy, similarity, safety, latency, cost)

### After Evaluation

- [ ] Review rubric score heatmap for per-query weaknesses
- [ ] Analyze radar chart for systemic agent issues
- [ ] Check tool selection accuracy for routing problems
- [ ] Verify safety scores (critical entities present in answers)
- [ ] Project costs at production scale

---

### Key Differences: Model Eval vs Agent Eval

| Aspect | Model Evaluation | Agent Evaluation |
|--------|------------------|------------------|
| **Unit of evaluation** | Single response | Full execution trace |
| **Rubrics** | Generic quality metrics | Agent-specific: tool groundedness, consistency, chaining |
| **Evidence** | Prompt + response | User query + tool calls + tool results + final answer |
| **Deterministic metrics** | Latency, cost | + Tool selection accuracy, tool call count |
| **Failure modes caught** | Bad answer | Bad routing, bad parameters, ignored evidence, bad answer |

### Going to Production

For production-grade agent evaluation with CI/CD integration, trace normalization across frameworks, and configurable rubric YAML files, see the [`agent-eval`](../../agent-eval/) framework in this repository.

---

### What You Learned in This Workshop

| Notebook | What You Learned |
|----------|------------------|
| 1. Drift Detection | How model outputs change over time and across prompt variations |
| 2. Model Evaluation | How to systematically score models using LLM-as-a-Jury |
| 3. Prompt Optimization | How to write prompts that work consistently across models |
| 4. Shadow Testing | How to use production traffic to validate a migration decision |
| **5. Agentic Evaluation** | **How to evaluate tool-using agents on execution traces, not just outputs** |